# Step 4 ? inspect and evaluate job requirements

This notebook uses only existing accepted, source-validated relevant AI pilot ads.
It never reclassifies jobs or calls an API. The subset is not a representative sample
of Swedish vacancies; technical acceptance and exact matching are not human accuracy.
Uncertain, irrelevant and failed classifications are kept in the registry outside
extraction. Missing requirements remain not_specified. Review flags do not block progress.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/extract_requirements.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.validate_classification import read_csv
from src.requirements_summary import evaluate_requirement_review
OUTPUT = ROOT/'data/processed/step4-requirements-pilot50-v2-20261023'
jobs = pd.DataFrame([json.loads(line) for line in (OUTPUT/'jobs_requirements.jsonl').read_text(encoding='utf-8').splitlines()])
facts = read_csv(OUTPUT/'requirements_evidence.csv')
summary = json.loads((OUTPUT/'requirements_summary.json').read_text(encoding='utf-8'))
run = json.loads((OUTPUT/'run_summary.json').read_text(encoding='utf-8'))
display(pd.Series({k:run[k] for k in ['available_accepted_relevant','selected_pilot_jobs','usable_requirements_jobs','extracted_requirement_records','expected_api_cost_usd']}))


## Classification coverage and extraction schema

All original records are preserved. We can analyze a useful subset without finishing
classification of every candidate. Accepted relevance is an AI decision, not human
verification. Research, governance and adjacent categories remain separate. Each
normalized evidence record may be a requested requirement, alternative, negation,
ambiguous qualification, mere mention, seniority or source metadata; do not count
all evidence rows as requested skills.


In [ ]:
display(pd.Series(run['classification_dispositions'], name='source_advertisement_count'))
display(pd.DataFrame({'column':jobs.columns}))
display(pd.DataFrame(summary['reporting_scopes']))
display(jobs[['advertisement_id','job_title','ai_primary_category','ai_prompt_version','classification_selected_run','requirements_data_usable']].head(12))


## Source-grounded evidence checks

Offsets are Unicode character offsets [start,end) into the original field. A
qualification heading may be separate from the entity's clause. These checks
prove exact text grounding, not that the rule interpreted the clause correctly.


In [ ]:
sources = jobs.set_index('advertisement_id').to_dict('index')
for row in facts.to_dict('records'):
    source = sources[row['advertisement_id']][row['source_field']]
    assert source[int(row['evidence_start']):int(row['evidence_end'])] == row['evidence_text']
    assert source[int(row['match_start']):int(row['match_end'])] == row['matched_text']
    if row['context_heading_start']:
        start, end = int(float(row['context_heading_start'])), int(float(row['context_heading_end']))
        assert source[start:end] == row['context_heading_text']
print(f'All {len(facts)} evidence records match original source slices.')
display(facts[['advertisement_id','kind','value','modality','rule_id','evidence_text','context_heading_text']].head(20))


## Requested skills and category differences

Counts are distinct advertisements, not repeated mentions. Required, preferred and
alternative-required facts stay separate; an alternative is not independently
mandatory. Mention-only/ambiguous technologies are excluded from requested counts.
The tables include selected-job and usable/skills-specified denominators.


In [ ]:
display(pd.Series(summary['denominators']))
display(pd.DataFrame(summary['requested_skills']).head(30))
display(pd.DataFrame(summary['skills_by_category']).head(30))


## Swedish and English requirements

not_specified means no sufficiently clear requirement was extracted, not that the
language is unnecessary. Explicit non-requirement, preference and requirement are
distinct. Contradictory/either-or evidence abstains and remains inspectable.


In [ ]:
for language, rows in summary['language_requirements'].items():
    print(language)
    display(pd.DataFrame(rows))
display(jobs[['advertisement_id','swedish_requirement','swedish_review_required','english_requirement','english_review_required']].head(15))


## Education, experience and seniority

Degrees are not inferred from titles. Multiple degree options remain multiple
levels. Literal years, numeric ranges and explicit minima stay distinct and retain
all contexts; study durations are not professional experience. Seniority is explicit
own-role wording, never inferred from age, years or senior colleagues.


In [ ]:
display(jobs.education_level.value_counts().rename_axis('education_level').reset_index(name='jobs'))
display(pd.DataFrame(summary['education_requirements']))
display(pd.DataFrame(summary['experience_requirements']))
display(pd.DataFrame(summary['seniority']))
display(jobs[['advertisement_id','education_levels','education_modalities','experience_requirements','seniority']].head(15))


## Employment and regional context

Region/municipality/employer/publication date come from original source fields.
Employment/time/remote facts are explicit source matches. Regional counts describe
this development subset only, with per-region selected and usable-job counts.


In [ ]:
display(pd.DataFrame(summary['regions']))
display(pd.DataFrame(summary['skills_by_region']).head(25))
display(jobs[['advertisement_id','region','municipality','employment_type','working_time','work_arrangement']].head(15))


## Manual review and evaluation

Read full descriptions alongside the normalized evidence. In the output folder,
edit only human_* fields and reviewer in manual_review_requirements.csv; mark
human_verified=true only after a named human reviews the fact. human_valid is
true, false or uncertain; human_modality records a manually checked modality.
Use manual_review_jobs.csv to record missed requirements independently. Review of
system outputs alone cannot establish recall. No model output or AI-assisted label
is automatically verified. The optional queue is not a requirement to fix every ad.


In [ ]:
review = read_csv(OUTPUT/'manual_review_requirements.csv')
metrics = evaluate_requirement_review(review)
display(pd.Series(metrics))
display(review[['requirement_id','advertisement_id','kind','value','modality','evidence_text','human_valid','human_modality','human_verified','reviewer']].head(15))


## Cost and optional ambiguity handling

The implemented pipeline is deterministic and costs $0 in API usage. The optional
LLM file is a provisional planning-only forecast for at most ten ambiguous ads;
there is no LLM execution backend or paid execution switch. Many missing facts
are genuinely absent. An optional future model must preserve evidence, abstain,
use a final schema/budget/checkpoints and receive separate execution approval.


In [ ]:
llm_plan = json.loads((OUTPUT/'optional_llm_plan.json').read_text(encoding='utf-8'))
display(pd.Series({k:v for k,v in llm_plan.items() if k!='requests'}))
assert run['classifier_requests']==0 and run['paid_extraction_requests']==0
print('No classification retries or paid extraction calls were made.')


## Full-advertisement drilldown

Change INSPECT_ID to inspect one complete original description alongside its
classification provenance and all extracted evidence. No API call or edit occurs.


In [ ]:
INSPECT_ID = jobs.iloc[0]['advertisement_id']
selected = jobs.loc[jobs.advertisement_id.eq(INSPECT_ID)].iloc[0]
print(selected['job_title'])
print(selected['description_text'])
display(facts.loc[facts.advertisement_id.eq(INSPECT_ID), ['kind','value','modality','evidence_start','evidence_end','evidence_text']])
